# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` as defined in the Croissant schema.

### Dataset Source
The dataset is defined by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", getattr(metadata, 'name', 'N/A'))
print("Description:", getattr(metadata, 'description', 'N/A'))


## 2. Data Overview

Let's enumerate available record sets, fields, and their `@id`s from the Croissant schema.

In [ ]:
# List available record sets and fields by @id

print("Available Record Sets (by @id):")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set.id} | name: {record_set.name}")
    print("    - Fields:")
    for field in record_set.fields:
        print(f"      - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print("")

# Save record set @ids in a list for later usage
record_set_ids = [rs.id for rs in dataset.record_sets]

## 3. Data Extraction

Load records from a selected record set by its `@id`, and create a DataFrame. You can change the `active_record_set_id` below to explore other record sets.

In [ ]:
# If there are multiple record sets, select one for demonstration
if len(record_set_ids) == 0:
    print("No record sets defined in the Croissant schema.")
else:
    # Choose the first record set, edit as appropriate
    active_record_set_id = record_set_ids[0]
    print(f"Using record set: {active_record_set_id}")

    records = list(dataset.records(record_set=active_record_set_id))
    df = pd.DataFrame(records)
    print(f"Fields (columns) in {active_record_set_id}:")
    print(list(df.columns))
    df.head()

## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group data by field `@id`. All operations reference column names by their field `@id` as per the Croissant schema.

*You can adjust field IDs below to match a numeric field and groupable field of interest from the previous step.*

In [ ]:
# Example: Choose a numeric field and a grouping field by their @ids
if len(record_set_ids) == 0:
    print("No record sets to analyze.")
else:
    # Use the previously selected record set and DataFrame
    columns = df.columns.tolist()
    # Guess potential numeric fields for demonstration
    import re
    numeric_field_id = None
    for col in columns:
        # Assume 'abundance' or 'MW' or 'peptide_count' as demo
        if re.search(r'(abundance|count|MW|pi|intensity|number|coverage|score)', col, re.I):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if numeric_field_id is None:
        # Pick the first column that is numeric
        for col in columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if numeric_field_id is None:
        print("No obvious numeric column found.")
    else:
        threshold = df[numeric_field_id].mean()
        print(f"Using numeric field '{numeric_field_id}' with threshold {threshold:.2f} (mean value)")

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean value):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Guess a group field (e.g., a categorical column)
        group_field_id = None
        for col in columns:
            if col != numeric_field_id and df[col].nunique() < 20 and pd.api.types.is_object_dtype(df[col]):
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping analysis.")

## 5. Visualization

Visualize the distribution of the selected numeric field, or the mean per group if a group field was used.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_set_ids) == 0 or numeric_field_id is None:
    print("No numeric field to visualize.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='teal')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.show()

    # If grouped by a categorical field
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id], palette='viridis')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion

In this notebook, we loaded and explored a mass spectrometry dataset using the Croissant schema and the `mlcroissant` library. We:
- Loaded metadata and records from the FAIR^2 Croissant dataset.
- Enumerated available record sets and fields by their `@id`.
- Loaded records into DataFrames using the `@id`s.
- Performed basic EDA using only field `@id` references, including filtering, normalization, and grouping.
- Visualized the distribution of a selected numeric field.

**Next steps:** Explore the dataset further by examining additional record sets, refining field selection, or extending the analyses according to your research needs.
